In [41]:
import re
import pandas as pd
import numpy as np
from scipy import stats

## Langkah 1 — Unggah data
Di Google Colab jalankan sel ini lalu pilih file `.xlsx`. Di komputer sendiri, isi `nama_file` dengan nama berkas.

In [ ]:
try:
    from google.colab import files
    up = files.upload()
    nama_file = list(up.keys())[0]
except ImportError:           # dijalankan di luar Colab
    nama_file = "Kuesioner_Penggunaan_AI_dan_Lifestyle_Media_Sosial_Mahasiswa_Jawaban.xlsx"   # <- ganti dengan nama file Anda
print("File terbaca:", nama_file)

Saving Blank Quiz (Responses 3 TIC) (1).xlsx to Blank Quiz (Responses 3 TIC) (1) (1).xlsx
File terbaca: Blank Quiz (Responses 3 TIC) (1) (1).xlsx


## Langkah 2 — Baca data dan pisahkan butir
Kolom yang judulnya berformat `Nama Dimensi [teks pernyataan]` dibaca sebagai **butir**. Kolom lain (Timestamp, Score, Kelas, Jenis Kelamin, jumlah platform AI, durasi gadget, dst.) dianggap data demografi dan tidak ikut diuji. Cara ini lebih aman daripada memotong lima kolom pertama, karena pada ekspor Google Forms ada kolom non-butir di tengah (mis. durasi gadget).

In [31]:
df = pd.read_excel(nama_file)

adalah_butir = [bool(re.match(r".*\[.*\]\s*$", str(k), flags=re.S)) for k in df.columns]
butir = df.loc[:, adalah_butir]
demografi = df.loc[:, [not a for a in adalah_butir]]

print("Jumlah responden :", len(df))
print("Jumlah butir     :", butir.shape[1])
print("Data kosong      :", int(butir.isna().sum().sum()))
print("Kolom bukan butir (tidak diuji):")
for k in demografi.columns:
    print("  -", k[:80])

Jumlah responden : 40
Jumlah butir     : 29
Data kosong      : 0
Kolom bukan butir (tidak diuji):
  - Timestamp
  - Kelas
  - Jenis Kelamin
  - Jumlah Platform AI yang rutin digunakan untuk membatu menyelesaikan tugas atau p
  - Jumlah Platform AI yang berlangganan premium atau tidak free (isi dengan num/ang
  - Durasi (jam) rata-rata penggunaan gadget perhari, untuk , kebutuhan harian sampa


## Langkah 3 — Ubah jawaban Likert menjadi angka
Sangat Setuju = 5 … Sangat Tidak Setuju = 1. Penulisan label dinormalkan (spasi dan huruf besar/kecil) supaya tidak ada sel yang gagal dibaca. Sel yang sudah berupa angka 1–5 dibiarkan.

In [32]:
skala = {"sangat tidak setuju": 1, "sts": 1,
         "tidak setuju": 2, "ts": 2,
         "netral": 3, "n": 3,
         "setuju": 4, "s": 4,
         "sangat setuju": 5, "ss": 5}

def ke_angka(v):
    if pd.isna(v):
        return np.nan
    if isinstance(v, (int, float, np.integer, np.floating)):
        return float(v)
    teks = re.sub(r"\s+", " ", str(v)).strip().lower()
    teks = re.sub(r"\s*\(.*?\)\s*$", "", teks)          # buang "(S)", "(SS)", dst.
    if teks in skala:
        return float(skala[teks])
    kode = re.search(r"\((.*?)\)\s*$", str(v))
    return float(skala.get(kode.group(1).strip().lower(), np.nan)) if kode else np.nan

num = butir.apply(lambda kol: kol.map(ke_angka))

# Harus 0. Jika lebih dari 0, ada jawaban yang tidak terbaca.
print("Sel yang gagal dikonversi:", int(num.isna().sum().sum()))
print("Rentang nilai:", num.min().min(), "sampai", num.max().max())
num.head()

Sel yang gagal dikonversi: 0
Rentang nilai: 1.0 sampai 5.0


,Penggunaan AI pada tugas atau projek kuliah [Saya menggunakan platform AI hampir setiap kali mendapatkan tugas akademik.],Penggunaan AI pada tugas atau projek kuliah [Saya menggunakan AI untuk mencari solusi atas permasalahan atau soal perkuliahan.],Penggunaan AI pada tugas atau projek kuliah [Saya menggunakan AI untuk mencari ide atau inspirasi dalam mengerjakan tugas./projek],Penggunaan AI pada tugas atau projek kuliah [Saya merasa kesulitan memulai mengerjakan tugas tanpa bantuan AI.],Penggunaan AI pada tugas atau projek kuliah [Saya merasa kurang yakin dengan hasil tugas apabila tidak menggunakan AI.],Penggunaan AI pada tugas atau projek kuliah [Saya percaya bahwa AI dapat memberikan informasi yang membantu penyelesaian tugas.],Penggunaan AI pada tugas atau projek kuliah [Saya memeriksa kembali kebenaran informasi yang diberikan AI sebelum menggunakannya.],Aspek Psikologis penggunaan AI pada tugas atau projek kuliah [Saya merasa perlu menggunakan AI setiap kali mendapatkan tugas atau proyek kuliah.],Aspek Psikologis penggunaan AI pada tugas atau projek kuliah [aya merasa kesulitan menyelesaikan tugas kuliah apabila tidak menggunakan bantuan AI.],Aspek Psikologis penggunaan AI pada tugas atau projek kuliah [Saya cenderung mengikuti saran atau jawaban AI tanpa banyak mempertimbangkan alternatif lainnya (Non-AI).],...,Kebiasaan penggunaan teknologi digital [Saya jarang memeriksa kembali hasil dari AI karena sudah percaya terhadap ketepatan jawabannya.],Prokrastinasi akademik. [Saya cenderung menunggu hingga mendekati deadline untuk mulai mengerjakan tugas.],Prokrastinasi akademik. [Saya sering menunda pengerjaan tugas karena merasa dapat dikerjakan dengan mudah dan cepat dengan menggunakan AI],Prokrastinasi akademik. [Saya sering menyelesaikan tugas kuliah dalam waktu yang sangat dekat dengan batas pengumpulan.],Prokrastinasi akademik. [Saya sering mengerjakan beberapa tugas sekaligus karena sebelumnya menunda pengerjaannya.],Respon emosional [Saya merasa tertekan ketika harus menyelesaikan tugas dalam waktu singkat akibat menunda pengerjaannya.],Respon emosional [Saya merasa panik ketika beberapa tugas yang saya tunda memiliki batas waktu pengumpulan yang berdekatan.],Respon emosional [Saya merasa kurang puas terhadap hasil tugas yang dikerjakan karena terburu-buru mendekati deadline.],Respon emosional [Saya merasa lega setelah berhasil mengumpulkan tugas meskipun pengerjaannya dilakukan AI saat mendekati deadline.],Respon emosional [Saya merasa menyesal karena hasil tugas mungkin bisa lebih baik apabila dikerjakan lebih awal.]
0,5.0,5.0,4.0,4.0,4.0,4.0,4.0,5.0,4.0,4.0,...,2.0,5.0,4.0,5.0,4.0,5.0,5.0,4.0,4.0,5.0
1,4.0,4.0,4.0,4.0,4.0,4.0,4.0,5.0,4.0,4.0,...,4.0,2.0,4.0,4.0,4.0,5.0,4.0,5.0,5.0,5.0
2,5.0,5.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,...,2.0,4.0,5.0,4.0,2.0,5.0,5.0,5.0,4.0,5.0
3,5.0,4.0,5.0,2.0,4.0,5.0,5.0,4.0,4.0,2.0,...,2.0,4.0,4.0,4.0,4.0,5.0,4.0,4.0,4.0,4.0
4,5.0,5.0,4.0,4.0,5.0,5.0,5.0,4.0,2.0,4.0,...,2.0,2.0,2.0,2.0,4.0,5.0,5.0,4.0,5.0,5.0


**Data numerik** disimpan ke Excel (kolom demografi + semua butir sudah berupa angka) sebagai bukti langkah konversi.

In [33]:
numerik_lengkap = pd.concat([demografi, num], axis=1)
numerik_lengkap.to_excel("data_numerik_likert.xlsx", index=False)
print("Tersimpan: data_numerik_likert.xlsx")

Tersimpan: data_numerik_likert.xlsx


## Langkah 4 — Kelompokkan butir per dimensi
Nama dimensi diambil dari teks sebelum tanda `[` (spasi dan titik di ujung dibuang; salah ketik diperbaiki lewat `KOREKSI_NAMA`).

In [34]:
KOREKSI_NAMA = {"Rsspon emosional": "Respon emosional"}      # perbaiki salah ketik pada judul dimensi

def nama_dimensi(kolom):
    m = re.match(r"^(.*?)\s*\[", kolom, flags=re.S)
    nama = (m.group(1) if m else "Semua butir").strip().rstrip(".").strip()
    return KOREKSI_NAMA.get(nama, nama)

def kelompokkan(num):
    dimensi = {}
    for kolom in num.columns:
        dimensi.setdefault(nama_dimensi(kolom), []).append(kolom)
    return dimensi

dimensi = kelompokkan(num)
for nama, daftar in dimensi.items():
    print(f"{nama}: {len(daftar)} butir")

Penggunaan AI pada tugas atau projek kuliah: 7 butir
Aspek Psikologis penggunaan AI pada tugas atau projek kuliah: 7 butir
Kebiasaan penggunaan teknologi digital: 6 butir
Prokrastinasi akademik: 4 butir
Respon emosional: 5 butir


## Langkah 5 — Uji validitas (korelasi Pearson butir–total)
- r tabel bergantung pada **n** dan **α = 0,05** (df = n − 2); untuk n = 30, r tabel = 0,361.
- r hitung **>** r tabel → **valid**; r hitung **≤** r tabel → **tidak valid**.
- *Corrected r* = korelasi butir dengan (total − butir itu sendiri), pembanding yang lebih ketat (batas 0,30).

In [35]:
def r_tabel_dari(n, alpha=0.05):
    t = stats.t.ppf(1 - alpha / 2, n - 2)
    return t / np.sqrt((n - 2) + t**2)

def uji_validitas(num, dimensi):
    r_tab = r_tabel_dari(len(num))
    baris = []
    for nama, daftar in dimensi.items():
        x = num[daftar]
        total = x.sum(axis=1)
        for i, kolom in enumerate(daftar, 1):
            r, p = stats.pearsonr(x[kolom], total)
            rk, _ = stats.pearsonr(x[kolom], total - x[kolom]) if len(daftar) > 1 else (np.nan, None)
            baris.append({"Dimensi": nama, "Butir": f"B{i}", "Kolom asli": kolom,
                          "r hitung": round(r, 3), "Corrected r": round(rk, 3),
                          "p-value": round(p, 4),
                          "Keterangan": "Valid" if r > r_tab else "Tidak valid"})
    return pd.DataFrame(baris), r_tab

tabel_validitas, r_tabel = uji_validitas(num, dimensi)
print(f"n = {len(num)} | r tabel (α=0,05; df={len(num)-2}) = {r_tabel:.3f}")
tabel_validitas.drop(columns="Kolom asli")

n = 40 | r tabel (α=0,05; df=38) = 0.312


,Dimensi,Butir,r hitung,Corrected r,p-value,Keterangan
0,Penggunaan AI pada tugas atau projek kuliah,B1,0.419,0.222,0.0071,Valid
1,Penggunaan AI pada tugas atau projek kuliah,B2,0.390,0.224,0.0129,Valid
2,Penggunaan AI pada tugas atau projek kuliah,B3,0.626,0.418,0.0000,Valid
3,Penggunaan AI pada tugas atau projek kuliah,B4,0.506,0.205,0.0009,Valid
4,Penggunaan AI pada tugas atau projek kuliah,B5,0.793,0.597,0.0000,Valid
5,Penggunaan AI pada tugas atau projek kuliah,B6,0.634,0.424,0.0000,Valid
6,Penggunaan AI pada tugas atau projek kuliah,B7,0.231,-0.061,0.1510,Tidak valid
7,Aspek Psikologis penggunaan AI pada tugas atau...,B1,0.578,0.437,0.0001,Valid
8,Aspek Psikologis penggunaan AI pada tugas atau...,B2,0.708,0.541,0.0000,Valid
9,Aspek Psikologis penggunaan AI pada tugas atau...,B3,0.804,0.651,0.0000,Valid


In [36]:
ringkas = tabel_validitas.groupby("Dimensi")["Keterangan"].value_counts().unstack(fill_value=0)
ringkas.loc["TOTAL"] = ringkas.sum()
print(ringkas)
print("\nButir tidak valid:")
tabel_validitas[tabel_validitas["Keterangan"] == "Tidak valid"].drop(columns="Kolom asli")

Keterangan                                          Tidak valid  Valid
Dimensi                                                               
Aspek Psikologis penggunaan AI pada tugas atau ...            1      6
Kebiasaan penggunaan teknologi digital                        0      6
Penggunaan AI pada tugas atau projek kuliah                   1      6
Prokrastinasi akademik                                        0      4
Respon emosional                                              0      5
TOTAL                                                         2     27

Butir tidak valid:


,Dimensi,Butir,r hitung,Corrected r,p-value,Keterangan
6,Penggunaan AI pada tugas atau projek kuliah,B7,0.231,-0.061,0.1510,Tidak valid
13,Aspek Psikologis penggunaan AI pada tugas atau...,B7,-0.094,-0.272,0.5625,Tidak valid


## Langkah 6 — Uji reliabilitas (Cronbach's alpha, per dimensi)
α ≥ 0,70 → reliabel; α < 0,70 → tidak reliabel (kategori lengkap: ≥ 0,90 sangat baik, 0,80–0,89 baik, 0,70–0,79 reliabel).
*Alpha jika butir dihapus* lebih tinggi dari α dimensi → butir itu melemahkan konsistensi.

In [37]:
def cronbach_alpha(x):
    k = x.shape[1]
    return k / (k - 1) * (1 - x.var(ddof=1).sum() / x.sum(axis=1).var(ddof=1))

def kategori(a):
    return ("Sangat baik" if a >= 0.90 else "Baik" if a >= 0.80
            else "Reliabel" if a >= 0.70 else "Tidak reliabel")

def uji_reliabilitas(num, dimensi):
    hasil, dihapus = [], []
    for nama, daftar in dimensi.items():
        x = num[daftar]
        a = cronbach_alpha(x)
        hasil.append({"Dimensi": nama, "Jumlah butir": len(daftar),
                      "Cronbach's alpha": round(a, 3), "Kategori": kategori(a)})
        for i, kolom in enumerate(daftar, 1):
            if len(daftar) > 2:
                dihapus.append({"Dimensi": nama, "Butir": f"B{i}",
                                "Alpha jika dihapus": round(cronbach_alpha(x.drop(columns=kolom)), 3)})
    return pd.DataFrame(hasil), pd.DataFrame(dihapus)

tabel_reliabilitas, tabel_alpha_dihapus = uji_reliabilitas(num, dimensi)
display(tabel_reliabilitas)
display(tabel_alpha_dihapus)

,Dimensi,Jumlah butir,Cronbach's alpha,Kategori
0,Penggunaan AI pada tugas atau projek kuliah,7,0.550,Tidak reliabel
1,Aspek Psikologis penggunaan AI pada tugas atau...,7,0.678,Tidak reliabel
2,Kebiasaan penggunaan teknologi digital,6,0.808,Baik
3,Prokrastinasi akademik,4,0.865,Baik
4,Respon emosional,5,0.555,Tidak reliabel


,Dimensi,Butir,Alpha jika dihapus
0,Penggunaan AI pada tugas atau projek kuliah,B1,0.531
1,Penggunaan AI pada tugas atau projek kuliah,B2,0.533
2,Penggunaan AI pada tugas atau projek kuliah,B3,0.456
3,Penggunaan AI pada tugas atau projek kuliah,B4,0.547
4,Penggunaan AI pada tugas atau projek kuliah,B5,0.341
5,Penggunaan AI pada tugas atau projek kuliah,B6,0.453
6,Penggunaan AI pada tugas atau projek kuliah,B7,0.639
7,Aspek Psikologis penggunaan AI pada tugas atau...,B1,0.637
8,Aspek Psikologis penggunaan AI pada tugas atau...,B2,0.597
9,Aspek Psikologis penggunaan AI pada tugas atau...,B3,0.550


## Langkah 7 — Tindakan untuk butir tidak valid
Butir yang tidak valid **dibuang**, lalu Langkah 4–6 diulang pada data yang tersisa. Pertimbangkan juga isi butirnya, jangan hanya mengejar angka.
Sel di bawah membuang semua butir tidak valid, lalu membandingkan alpha sebelum dan sesudah.

In [38]:
gabung = tabel_validitas.merge(tabel_alpha_dihapus, on=["Dimensi", "Butir"], how="left")
dibuang = tabel_validitas.loc[tabel_validitas["Keterangan"] == "Tidak valid", "Kolom asli"].tolist()
print("Butir dibuang:", len(dibuang))

num2 = num.drop(columns=dibuang)
dimensi2 = kelompokkan(num2)
tabel_validitas2, r_tabel2 = uji_validitas(num2, dimensi2)
tabel_reliabilitas2, tabel_alpha_dihapus2 = uji_reliabilitas(num2, dimensi2)

print("Validitas setelah butir dibuang (semua harus Valid):")
print(tabel_validitas2["Keterangan"].value_counts().to_string())
perbandingan = tabel_reliabilitas.merge(tabel_reliabilitas2, on="Dimensi", how="left",
                                        suffixes=(" (sebelum)", " (sesudah)"))
perbandingan[["Dimensi", "Jumlah butir (sebelum)", "Cronbach's alpha (sebelum)",
              "Jumlah butir (sesudah)", "Cronbach's alpha (sesudah)", "Kategori (sesudah)"]]

Butir dibuang: 2
Validitas setelah butir dibuang (semua harus Valid):
Keterangan
Valid    27


,Dimensi,Jumlah butir (sebelum),Cronbach's alpha (sebelum),Jumlah butir (sesudah),Cronbach's alpha (sesudah),Kategori (sesudah)
0,Penggunaan AI pada tugas atau projek kuliah,7,0.550,6,0.639,Tidak reliabel
1,Aspek Psikologis penggunaan AI pada tugas atau...,7,0.678,6,0.769,Reliabel
2,Kebiasaan penggunaan teknologi digital,6,0.808,6,0.808,Baik
3,Prokrastinasi akademik,4,0.865,4,0.865,Baik
4,Respon emosional,5,0.555,5,0.555,Tidak reliabel


## Langkah 8 — Latihan modul: corrected r dan ukuran sampel

In [39]:
# Latihan 2: jumlah butir valid menurut r hitung vs corrected r >= 0,30
v_r = int((tabel_validitas["r hitung"] > r_tabel).sum())
v_c = int((tabel_validitas["Corrected r"] >= 0.30).sum())
print(f"Valid menurut r hitung > r tabel : {v_r} dari {len(tabel_validitas)}")
print(f"Valid menurut corrected r >= 0,30: {v_c} dari {len(tabel_validitas)}")

# Latihan 4: pengaruh ukuran sampel (subset acak 20 responden)
sub = num.sample(20, random_state=1)
tv_sub, rt_sub = uji_validitas(sub, dimensi)
print(f"\nSubset n=20: r tabel = {rt_sub:.3f}, butir valid = {(tv_sub['Keterangan']=='Valid').sum()} dari {len(tv_sub)}")

Valid menurut r hitung > r tabel : 27 dari 29
Valid menurut corrected r >= 0,30: 21 dari 29

Subset n=20: r tabel = 0.444, butir valid = 23 dari 29


## Langkah 9 — Simpan hasil sebagai lampiran

In [40]:
with pd.ExcelWriter("hasil_uji_validitas_reliabilitas.xlsx") as w:
    gabung.drop(columns="Kolom asli").to_excel(w, sheet_name="Validitas", index=False)
    tabel_reliabilitas.to_excel(w, sheet_name="Reliabilitas", index=False)
    tabel_validitas2.drop(columns="Kolom asli").to_excel(w, sheet_name="Validitas setelah dibuang", index=False)
    perbandingan.to_excel(w, sheet_name="Alpha sebelum-sesudah", index=False)
try:
    files.download("hasil_uji_validitas_reliabilitas.xlsx")
    files.download("data_numerik_likert.xlsx")
except NameError:
    print("Tersimpan di folder kerja.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>